# 使用 Langchain、AnalyticDB 和 OpenAI 进行问答
本 notebook 介绍如何使用 Langchain、作为知识库的 AnalyticDB 以及 OpenAI 嵌入来实现一个问答系统。如果你还不熟悉 AnalyticDB，最好先看看 [Getting_started_with_AnalyticDB_and_OpenAI.ipynb](Getting_started_with_AnalyticDB_and_OpenAI.ipynb) notebook。

本 notebook 展示了一个端到端的流程：
- 使用 OpenAI API 计算嵌入。
- 将嵌入存储到 AnalyticDB 实例中以构建知识库。
- 使用 OpenAI API 将原始文本查询转换为嵌入。
- 使用 AnalyticDB 在创建的集合中执行最近邻搜索以找到相关上下文。
- 让 LLM 在给定的上下文中找出答案。

所有这些步骤都将简化为调用相应的 Langchain 方法。

## 前提条件
为完成本练习，我们需要准备以下几样东西：
[AnalyticDB 云实例](https://www.alibabacloud.com/help/en/analyticdb-for-postgresql/latest/product-introduction-overview)。
作为框架的 [Langchain](https://github.com/hwchase17/langchain)。
一个 OpenAI API 密钥。

### 安装依赖
本 notebook 需要以下 Python 包：`openai`、`tiktoken`、`langchain` 和 `psycopg2cffi`。
- `openai` 提供对 OpenAI API 的便捷访问。
- `tiktoken` 是一个用于 OpenAI 模型的快速 BPE 分词器。
- `langchain` 帮助我们更轻松地构建基于 LLM 的应用。
- `psycopg2cffi` 库用于与向量数据库交互，但任何其他 PostgreSQL 客户端库也都可以。

In [ ]:
! pip install openai tiktoken langchain psycopg2cffi 

In [2]:
! export OPENAI_API_KEY="your API key"

In [1]:
# Test that your OpenAI API key is correctly set as an environment variable
# Note. if you run this notebook locally, you will need to reload your terminal and the notebook for the env variables to be live.
import os

# Note. alternatively you can set a temporary env variable like this:
# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

if os.getenv("OPENAI_API_KEY") is not None:
    print("OPENAI_API_KEY is ready")
else:
    print("OPENAI_API_KEY environment variable not found")

OPENAI_API_KEY is ready


### 准备 OpenAI API 密钥
OpenAI API 密钥用于对文档和查询进行向量化。

如果你还没有 OpenAI API 密钥，可以从 [https://platform.openai.com/account/api-keys ) 获取。

拿到密钥后，请运行以下命令将其添加到环境变量 `OPENAI_API_KEY` 中：

### 准备 AnalyticDB 连接字符串
要构建 AnalyticDB 连接字符串，你需要以下参数：`PG_HOST`、`PG_PORT`、`PG_DATABASE`、`PG_USER` 和 `PG_PASSWORD`。你需要先导出它们以设置正确的连接字符串，然后再构建连接字符串。

In [4]:
! export PG_HOST="your AnalyticDB host url"
! export PG_PORT=5432 # Optional, default value is 5432
! export PG_DATABASE=postgres # Optional, default value is postgres
! export PG_USER="your username"
! export PG_PASSWORD="your password"

In [2]:
import os
from langchain.vectorstores.analyticdb import AnalyticDB

CONNECTION_STRING = AnalyticDB.connection_string_from_db_params(
    driver=os.environ.get("PG_DRIVER", "psycopg2cffi"),
    host=os.environ.get("PG_HOST", "localhost"),
    port=int(os.environ.get("PG_PORT", "5432")),
    database=os.environ.get("PG_DATABASE", "postgres"),
    user=os.environ.get("PG_USER", "postgres"),
    password=os.environ.get("PG_PASSWORD", "postgres"),
)

In [3]:
import json

with open("questions.json", "r") as fp:
    questions = json.load(fp)

with open("answers.json", "r") as fp:
    answers = json.load(fp)

## 加载数据
在本节中，我们将加载包含一些自然问题及其答案的数据。所有数据都将用于创建一个以 AnalyticDB 为知识库的 Langchain 应用。

In [4]:
print(questions[0])

when is the last episode of season 8 of the walking dead


In [ ]:
import wget

# All the examples come from https://ai.google.com/research/NaturalQuestions
# This is a sample of the training set that we download and extract for some
# further processing.
wget.download("https://storage.googleapis.com/dataset-natural-questions/questions.json")
wget.download("https://storage.googleapis.com/dataset-natural-questions/answers.json")

In [6]:
print(answers[0])

No . overall No. in season Title Directed by Written by Original air date U.S. viewers ( millions ) 100 `` Mercy '' Greg Nicotero Scott M. Gimple October 22 , 2017 ( 2017 - 10 - 22 ) 11.44 Rick , Maggie , and Ezekiel rally their communities together to take down Negan . Gregory attempts to have the Hilltop residents side with Negan , but they all firmly stand behind Maggie . The group attacks the Sanctuary , taking down its fences and flooding the compound with walkers . With the Sanctuary defaced , everyone leaves except Gabriel , who reluctantly stays to save Gregory , but is left behind when Gregory abandons him . Surrounded by walkers , Gabriel hides in a trailer , where he is trapped inside with Negan . 101 `` The Damned '' Rosemary Rodriguez Matthew Negrete & Channing Powell October 29 , 2017 ( 2017 - 10 - 29 ) 8.92 Rick 's forces split into separate parties to attack several of the Saviors ' outposts , during which many members of the group are killed ; Eric is critically injure

## 定义链

Langchain 已经与 AnalyticDB 集成，并会为给定的文档列表完成所有索引工作。在我们的例子中，我们将存储已有的答案集合。

In [7]:
from langchain.vectorstores import AnalyticDB
from langchain.embeddings import OpenAIEmbeddings
from langchain import VectorDBQA, OpenAI

embeddings = OpenAIEmbeddings()
doc_store = AnalyticDB.from_texts(
    texts=answers, embedding=embeddings, connection_string=CONNECTION_STRING,
    pre_delete_collection=True,
)

此时所有可能的答案都已存储在 AnalyticDB 中，因此我们可以定义完整的问答链。

In [9]:
from langchain.chains import RetrievalQA

llm = OpenAI()
qa = VectorDBQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    vectorstore=doc_store,
    return_source_documents=False,
)

## 搜索数据

数据放入 AnalyticDB 之后，我们就可以开始提问了。问题会被 OpenAI 模型自动向量化，生成的向量将用于在 AnalyticDB 中查找可能匹配的答案。检索到之后，最相似的答案会被整合到发送给 OpenAI 大语言模型的提示中。


In [10]:
import random

random.seed(52)
selected_questions = random.choices(questions, k=5)

In [11]:
for question in selected_questions:
    print(">", question)
    print(qa.run(question), end="\n\n")

> where do frankenstein and the monster first meet
 Victor retreats into the mountains, and that is where the Creature finds him and pleads for Victor to hear his tale.

> who are the actors in fast and furious
 The main cast of Fast & Furious includes Vin Diesel as Dominic Toretto, Paul Walker as Brian O'Conner, Michelle Rodriguez as Letty Ortiz, Jordana Brewster as Mia Toretto, Tyrese Gibson as Roman Pearce, and Ludacris as Tej Parker.

> properties of red black tree in data structure
 The properties of a red-black tree in data structure are that each node is either red or black, the root is black, all leaves (NIL) are black, and if a node is red, then both its children are black. Additionally, every path from a given node to any of its descendant NIL nodes contains the same number of black nodes.

> who designed the national coat of arms of south africa
 Iaan Bekker

> caravaggio's death of the virgin pamela askew
 I don't know.



### 自定义提示模板

Langchain 中的 `stuff` 链类型使用一个整合了问题和上下文文档的特定提示。默认提示如下所示：

```text
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.
{context}
Question: {question}
Helpful Answer:
```

不过，我们可以提供自己的提示模板来改变 OpenAI LLM 的行为，同时仍然使用 `stuff` 链类型。重要的是要保留 `{context}` 和 `{question}` 这两个占位符。

#### 尝试自定义提示

我们可以尝试使用不同的提示模板，让模型：
1. 如果知道答案，就用一句话回答。
2. 如果不知道我们问题的答案，就随机推荐一首歌名。

In [12]:
from langchain.prompts import PromptTemplate
custom_prompt = """
Use the following pieces of context to answer the question at the end. Please provide
a short single-sentence summary answer only. If you don't know the answer or if it's
not present in given context, don't try to make up an answer, but suggest me a random
unrelated song title I could listen to.
Context: {context}
Question: {question}
Helpful Answer:
"""

custom_prompt_template = PromptTemplate(
    template=custom_prompt, input_variables=["context", "question"]
)

In [13]:
custom_qa = VectorDBQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    vectorstore=doc_store,
    return_source_documents=False,
    chain_type_kwargs={"prompt": custom_prompt_template},
)

In [14]:
random.seed(41)
for question in random.choices(questions, k=5):
    print(">", question)
    print(custom_qa.run(question), end="\n\n")

> what was uncle jesse's original last name on full house
Uncle Jesse's original last name on Full House was Cochran.

> when did the volcano erupt in indonesia 2018
No information about a volcano erupting in Indonesia in 2018 is present in the given context. Suggested song title: "Volcano" by U2.

> what does a dualist way of thinking mean
A dualist way of thinking means believing that humans possess a non-physical mind or soul which is distinct from their physical body.

> the first civil service commission in india was set up on the basis of recommendation of
The first Civil Service Commission in India was not set up on the basis of a recommendation.

> how old do you have to be to get a tattoo in utah
In Utah, you must be at least 18 years old to get a tattoo.

